# Fine-tune EnViT5 bằng LoRA r=32 trên Google Colab

Chọn runtime GPU, mở notebook này và chạy từ đầu. Checkpoint và bản dịch được lưu trên Google Drive. Nếu phiên bị ngắt, chạy lại các ô để tiếp tục từ checkpoint gần nhất (mỗi 500 bước optimizer). Nếu đã lưu xong adapter tốt nhất, notebook chuyển sang giải mã. Đổi tên thư mục thí nghiệm khi thay cấu hình.

In [ ]:
import subprocess
import sys
from pathlib import Path

import google.colab.drive
google.colab.drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "peft==0.17.1", "accelerate>=1,<2",
                "datasets>=3,<5", "sentencepiece", "tensorboard", "sacrebleu>=2.4,<3"], check=True)

## Tải IWSLT’15 English–Vietnamese
Notebook tải trực tiếp các tệp nén của corpus Stanford IWSLT’15 từ bản sao cố định, kiểm tra SHA-256 của sáu tệp Anh/Việt và ghép từng cặp theo số dòng. Dùng train để fine-tune, tst2012 để chọn checkpoint và tst2013 để chấm BLEU. Drive dùng để lưu kết quả. [Trang dữ liệu Stanford](https://nlp.stanford.edu/projects/nmt/) · [Bản sao dùng trong notebook](https://github.com/stefan-it/nmt-en-vi/tree/93193ddf269449c98495e3455b2bb261b4851234/data)

In [ ]:
import hashlib
import io
from pathlib import Path
import tarfile
from tempfile import NamedTemporaryFile
from urllib.request import urlopen


DATA = Path("/content/data/raw/iwslt15")
BASE = "https://raw.githubusercontent.com/stefan-it/nmt-en-vi/93193ddf269449c98495e3455b2bb261b4851234/data"
ARCHIVES = {"train": "train-en-vi", "tst2012": "dev-2012-en-vi", "tst2013": "test-2013-en-vi"}
CHECKSUMS = {
    "train.en": "c26dfeed74b6bf3752f5ca552f2412456f0de153f7c804df8717931fb3a5c78a",
    "train.vi": "707206edf2dc0280273952c7b70544ea8a1363aa69aaeb9d70514b888dc3067d",
    "tst2012.en": "b8fbcd0d4199d41276421d1f7228298df67dd9e2c5c134c2ee8cdf3ec27801e2",
    "tst2012.vi": "01004078b3f36b0c46e81b65fc6851230b3c823405c76d1d82ef3b786b512ff0",
    "tst2013.en": "3860ce60bc5b85d96cc2d3d580cf1bc9d62440e939c1b964676d3607c4e8f1df",
    "tst2013.vi": "29af16842808da2812b8fd4d47fc05d082ba870e60ab7755090ad04b380994cd",
}


def download():
    DATA.mkdir(parents=True, exist_ok=True)
    for split, archive in ARCHIVES.items():
        paths = [DATA / f"{split}.{lang}" for lang in ("en", "vi")]
        missing = [p for p in paths if not p.exists() or hashlib.sha256(p.read_bytes()).hexdigest() != CHECKSUMS[p.name]]
        if missing:
            with urlopen(f"{BASE}/{archive}.tgz", timeout=90) as response:
                compressed = io.BytesIO(response.read())
            with tarfile.open(fileobj=compressed, mode="r:gz") as bundle:
                for path in missing:
                    content = bundle.extractfile(path.name).read()
                    if hashlib.sha256(content).hexdigest() != CHECKSUMS[path.name]:
                        raise ValueError(f"Invalid SHA-256 for {path.name}")
                    with NamedTemporaryFile(dir=DATA, delete=False) as tmp:
                        tmp.write(content)
                    Path(tmp.name).replace(path)
        counts = [len(p.read_text(encoding="utf-8").splitlines()) for p in paths]
        if counts[0] != counts[1]:
            raise ValueError(f"Unaligned files: {split}")
        print(f"{split}: {counts[0]:,} pairs (SHA-256 OK)")

import html
import unicodedata

SPLITS = {"train": "train", "validation": "tst2012", "test": "tst2013"}


def normalize(text):
    return " ".join(unicodedata.normalize("NFC", html.unescape(text)).split())


def records(split):
    stem = SPLITS[split]
    with (DATA / f"{stem}.en").open(encoding="utf8") as en, (DATA / f"{stem}.vi").open(encoding="utf8") as vi:
        for row, (source, target) in enumerate(zip(en, vi, strict=True), 1):
            source, target = normalize(source), normalize(target)
            if split == "train" and (not source or not target):
                continue
            yield {"id": f"{stem}-{row}", "en": source, "vi": target}

download()
expected = {"train": 133166, "validation": 1553, "test": 1268}
for split, count in expected.items():
    actual = sum(1 for _ in records(split))
    if actual != count:
        raise ValueError(f"{split}: expected {count} pairs, found {actual}")
    print(f"{split}: {actual:,} aligned examples")

In [ ]:
import json
import time
import torch
import transformers
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from datasets import Dataset
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint
assert torch.cuda.is_available(), "Chọn GPU trong Runtime > Change runtime type"
set_seed(42)
model_id = "VietAI/envit5-translation"
revision = "840bc88104d5a4277af740eaedb024df8c3093e7"
output = Path("/content/drive/MyDrive/BTL_DeepLearning/artifacts")
run = output / "envit5-lora-r32"
run.mkdir(parents=True, exist_ok=True)
best = run / "best"
limit = 128

def save_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf8")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q", "v"], bias="none", revision=revision,
)

## Học có giám sát

Đóng băng trọng số gốc của EnViT5. Chỉ học hai ma trận LoRA hạng 32 tại các phép chiếu query và value trong self-attention của encoder, decoder và cross-attention của decoder. Dùng alpha 64, dropout 0,05. Huấn luyện bằng teacher forcing và cross-entropy dự đoán token tiếng Việt kế tiếp. Tokenizer gốc mã hóa nguồn có tiền tố `en:` và nhãn có tiền tố `vi:`. Collator đệm nhãn bằng -100. Batch 4 và tích lũy 8 bước tương đương 32 câu mỗi lần cập nhật trên một GPU. Đánh giá và lưu mỗi 500 bước, lấy checkpoint có validation loss thấp nhất. Mô hình khởi đầu đã được VietAI huấn luyện dịch trên MTet và PhoMT. [Model card](https://huggingface.co/VietAI/envit5-translation)

In [ ]:
def prepare(split, tokenizer):
    data = Dataset.from_list(list(records(split)))
    def encode(batch):
        result = {}
        for lang, key in (("en", "input_ids"), ("vi", "labels")):
            ids = tokenizer([lang + ": " + s for s in batch[lang]], truncation=False)["input_ids"]
            result["clipped_" + lang] = [len(seq) > limit for seq in ids]
            result[key] = [seq if len(seq) <= limit else seq[:limit-1] + [tokenizer.eos_token_id] for seq in ids]
        return result
    data = data.map(encode, batched=True, remove_columns=data.column_names)
    stats = {lang: sum(data["clipped_" + lang]) for lang in ("en", "vi")}
    stats["examples"] = len(data)
    return data.remove_columns(["clipped_en", "clipped_vi"]), stats

if not (best / "complete.json").exists():
    tok = AutoTokenizer.from_pretrained(model_id, revision=revision)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id, revision=revision)
    model = get_peft_model(model, lora_config)
    model.config.use_cache = False
    trainable = [name for name, p in model.named_parameters() if p.requires_grad]
    assert trainable and all(".lora_" in name for name in trainable)
    model.print_trainable_parameters()
    train, train_stats = prepare("train", tok)
    valid, val_stats = prepare("validation", tok)
    collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100)
    probe = collator([{"input_ids": [2, 1], "labels": [2, 1]}, {"input_ids": [1], "labels": [1]}])
    assert probe["labels"][1, -1].item() == -100
    args = Seq2SeqTrainingArguments(
        output_dir=str(run), learning_rate=3e-5, num_train_epochs=3, seed=42,
        per_device_train_batch_size=4, per_device_eval_batch_size=4, gradient_accumulation_steps=8,
        eval_strategy="steps", eval_steps=500, save_strategy="steps", save_steps=500,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        bf16=torch.cuda.is_bf16_supported(), gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}, label_names=["labels"],
        save_total_limit=2, report_to=["tensorboard"])
    save_json(run / "config.json", dict(model=model_id, revision=revision, train=train_stats,
              validation=val_stats, torch=str(torch.__version__), transformers=transformers.__version__,
              device=torch.cuda.get_device_name(), parameters=sum(p.numel() for p in model.parameters()),
              trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
              max_length=limit, arguments=args.to_dict()))
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train, eval_dataset=valid,
                             processing_class=tok, data_collator=collator)
    trainer.train(resume_from_checkpoint=get_last_checkpoint(str(run)))
    trainer.save_model(str(best))
    tok.save_pretrained(best)
    save_json(best / "complete.json", dict(checkpoint=trainer.state.best_model_checkpoint,
              validation_loss=trainer.state.best_metric))
    del trainer, model, collator, probe
    import gc
    gc.collect()
    torch.cuda.empty_cache()

## Đánh giá cùng checkpoint

Nạp tokenizer và adapter tốt nhất từ Drive, rồi gắn adapter vào đúng revision của EnViT5 gốc. Với từng decoder, ghi dự đoán theo ID gốc, BLEU và chữ ký SacreBLEU, số câu chạm giới hạn, tốc độ và kích thước batch. Thư mục predictions/test có cùng cấu trúc với notebook 05.

In [ ]:
from sacrebleu.metrics import BLEU
tok = AutoTokenizer.from_pretrained(best)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_id, revision=revision)
model = PeftModel.from_pretrained(base_model, best).to("cuda").eval()
@torch.inference_mode()
def evaluate(split, width):
    rows = list(records(split))
    estimates, capped = [], 0
    torch.cuda.synchronize()
    started = time.perf_counter()
    for first in range(0, len(rows), 4):
        inputs = tok(["en: " + r["en"] for r in rows[first:first+4]], padding=True,
                     truncation=True, max_length=limit, return_tensors="pt").to("cuda")
        ids = model.generate(**inputs, max_new_tokens=limit, do_sample=False, num_beams=width,
                             length_penalty=1.0)
        capped += (~ids[:, 1:].eq(tok.eos_token_id).any(dim=1)).sum().item()
        estimates.extend(s.strip().removeprefix("vi:").strip() for s in tok.batch_decode(ids, skip_special_tokens=True))
    seconds = time.perf_counter() - started
    decoder = "greedy" if width == 1 else "beam4"
    target = output / "predictions" / split / f"envit5-lora-r32-{decoder}-seed42.jsonl"
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.with_suffix(".tmp").open("w", encoding="utf8") as stream:
        for row, pred in zip(rows, estimates, strict=True):
            print(json.dumps(dict(id=row["id"], prediction=pred), ensure_ascii=False), file=stream)
    target.with_suffix(".tmp").replace(target)
    metric = BLEU(tokenize="13a", lowercase=False)
    score = metric.corpus_score(estimates, [[r["vi"] for r in rows]])
    stats = dict(bleu=score.score, signature=str(metric.get_signature()), examples=len(rows),
                 capped=capped, sentences_per_second=len(rows)/seconds, inference_batch=4, checkpoint=str(best))
    save_json(run / f"{split}-{decoder}.json", stats)
    return stats

for split in ("validation", "test"):
    for width in (1, 4):
        print(split, width, evaluate(split, width))